# Carnet de démarrage — sujet 02 · Favorita

**Objectif du sujet** : prévoir les ventes d'un article, par magasin, sur 16 jours.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

12,2 M de lignes. `horizon.csv` est une grille dense, sans cible. Les ventes négatives sont des retours.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **https://drive.google.com/drive/folders/1N1EJk5sY8cJ3PFtipJFbMFJIG2AqUYYl?usp=sharing** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_02_Favorita/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.

## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `horizon.csv` | à prédire — la cible est absente |
| `items.csv` | entraînement ou annexe |
| `stores.csv` | entraînement ou annexe |
| `holidays_events.csv` | entraînement ou annexe |
| `oil.csv` | entraînement ou annexe |
| `transactions.csv` | entraînement ou annexe |

**Format de sortie attendu** : `id,unit_sales`.
**Colonne d'identifiant** : `id`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.

## Sur les données lourdes — et pourquoi pas « raylib »

`raylib` est une bibliothèque **graphique** (fenêtres, dessin, jeux) : elle ne lit aucune table. Elle
n'a rien à faire ici. Les outils qui répondent au besoin sont :

| Outil | Rôle | Présent par défaut ? |
|---|---|---|
| **Parquet + pyarrow** | format colonne compressé, lecture sélective | **oui** avec pandas récent |
| **Polars** | moteur colonne, `scan_csv` paresseux | non — `pip install polars` |
| **DuckDB** | SQL directement sur CSV/Parquet, sans chargement | non — `pip install duckdb` |

Ce carnet utilise **Parquet** (garanti), et le remplace par Polars ou DuckDB si vous les installez. Le
principe ne change pas : **ne jamais charger ce dont on n'a pas besoin**, et **payer la conversion
CSV → Parquet une seule fois**.

In [ ]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = "__LIEN__"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"

In [ ]:
from pathlib import Path
import os

# Définit DONNEES_PROJET seulement si le dossier existe sur cette machine
dossier = Path.cwd() / "Projet_app_auto"
if dossier.exists():
    os.environ["DONNEES_PROJET"] = str(dossier)

In [ ]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'horizon.csv', 'items.csv', 'stores.csv', 'holidays_events.csv', 'oil.csv', 'transactions.csv']
SUJET = "Sujet_02_Favorita"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])

In [ ]:
# --- 3. Chargement efficace (table lourde) ------------------------------------
# Le cache Parquet s'écrit à côté du carnet, jamais dans les données : sur Kaggle,
# /kaggle/input est en lecture seule.
import pyarrow as pa
import pyarrow.csv as pvc
import pyarrow.parquet as pq


def table(nom, colonnes=None, complet=False):
    """Ouvre un CSV en profitant d'un cache Parquet.

    nom      : nom du fichier dans le dossier de données
    colonnes : liste de colonnes à garder (None = toutes) — le levier principal
    morceaux : si un entier, rend un itérateur de DataFrames de cette taille
    """
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")

    # lecture d'un échantillon : pas de cache, on veut seulement voir
    if ECHANTILLON and not complet:
        lots, reste = [], ECHANTILLON
        for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 25)):
            lots.append(lot)
            reste -= lot.num_rows
            if reste <= 0:
                break
        t = pa.Table.from_batches(lots)
        if colonnes:
            t = t.select([c for c in colonnes if c in t.column_names])
        return t.to_pandas()

    # cache Parquet, construit une fois
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    if not cache.is_file():
        DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)
        print(f"construction du cache Parquet : {cache.name} (une seule fois)")
        ecrivain = None
        try:
            for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 26)):
                if colonnes:
                    lot = lot.select([c for c in colonnes if c in lot.schema.names])
                if ecrivain is None:
                    ecrivain = pq.ParquetWriter(cache, lot.schema)
                ecrivain.write_batch(lot)
        finally:
            if ecrivain is not None:
                ecrivain.close()
    fichier = pq.ParquetFile(cache)
    return fichier.read(columns=colonnes).to_pandas()


def table_par_morceaux(nom, colonnes=None, morceaux=200_000):
    """Itère sur la table par paquets — pour les traitements qui ne tiennent pas en mémoire."""
    table(nom, colonnes=colonnes)          # garantit que le cache existe
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    for lot in pq.ParquetFile(cache).iter_batches(batch_size=morceaux):
        yield lot.to_pandas()

In [ ]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()

In [ ]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "unit_sales"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")

In [ ]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['id', 'unit_sales']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"id": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["unit_sales"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("horizon.csv", colonnes=["id"], complet=True)
identifiants = a_predire["id"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)

In [ ]:
# --- 7. Le travail commence ICI -----------------------------------------------
# À vous de décider, et de justifier :
#   - la métrique, et ce qu'elle coûte ;
#   - le découpage de validation ;
#   - les variables, leur encodage, leur imputation ;
#   - le modèle, son réglage, son seuil.
# Ce carnet ne le fera pas à votre place : c'est l'objet du projet.
#raise NotImplementedError(
#    "Carnet de démarrage : la modélisation est à votre charge. Écrivez votre pipeline à partir d'ici."
#)

# Séance 1
# Nom : HOUNKANLIN Fernand
# Porteur : P1
# Etapes Assignées : 1.1 -------- 1.8

# 1.1  Inventorier les tables
Ce document répertorie et structure l'ensemble des fichiers de données utilisés dans le notebook pour l'analyse et la modélisation.

---

##  Vue d'ensemble des fichiers

| Nom du Fichier | Type de Données | Description Générale |
| :--- | :--- | :--- |
| `train.csv` | Temporel / Cible | Données historiques d'entraînement (ventes, cibles). |
| `horizon.csv` | Temporel / Prédiction | Fenêtre de prédiction future ou variables exogènes projetées. |
| `items.csv` | Métadonnées | Informations et caractéristiques propres aux articles/produits. |
| `stores.csv` | Métadonnées | Informations géographiques et typologiques sur les magasins. |
| `holidays_events.csv` | Calendrier | Événements spéciaux, jours fériés et événements locaux/nationaux. |
| `oil.csv` | Économique | Suivi quotidien du prix du pétrole (variable exogène). |
| `transactions.csv` | Activité | Volume quotidien des transactions par magasin. |

---

##  Détail des Tables

### 1. `train.csv`
* **Rôle** : Table principale d'entraînement.
* **Colonnes clés attendues** : `date`, `store_nbr`, `item_nbr`, `sales`, `onpromotion`.
* **Description** : Contient l'historique des ventes par article et par magasin, ainsi que l'état promotionnel.

### 2. `horizon.csv`
* **Rôle** : Définition de la période de test ou des prévisions.
* **Description** : Structure la chronologie future à prédire avec les variables connues à l'avance.

### 3. `items.csv`
* **Rôle** : Dimension Produits.
* **Colonnes clés attendues** : `item_nbr`, `family`, `class`, `perishable`.
* **Description** : Regroupe les articles par famille et par classe, et indique s'ils sont périssables.

### 4. `stores.csv`
* **Rôle** : Dimension Magasins.
* **Colonnes clés attendues** : `store_nbr`, `city`, `state`, `type`, `cluster`.
* **Description** : Caractérise la localisation des points de vente et leur regroupement par similitude (`cluster`).

### 5. `holidays_events.csv`
* **Rôle** : Données contextuelles / Calendrier.
* **Colonnes clés attendues** : `date`, `type`, `locale`, `locale_name`, `description`, `transferred`.
* **Description** : Permet de capter l'impact des jours fériés ou des événements nationaux et locaux sur le comportement d'achat.

### 6. `oil.csv`
* **Rôle** : Indicateur macroéconomique.
* **Colonnes clés attendues** : `date`, `dcoilwtico`.
* **Description** : Données sur le prix quotidien du pétrole (brut léger WTI), crucial pour analyser les contextes économiques locaux.

### 7. `transactions.csv`
* **Rôle** : Comportement global du magasin.
* **Colonnes clés attendues** : `date`, `store_nbr`, `transactions`.
* **Description** : Indique le nombre de tickets de caisse ou de transactions globales par jour et par magasin (fortement corrélé aux volumes de ventes).

In [ ]:
# 1.1 — Inventaire des fichiers, dans le dossier trouvé par DOSSIER
from pathlib import Path
import pandas as pd

inventaire = []
for chemin in sorted(DOSSIER.glob('*.csv')):
    en_tete = pd.read_csv(chemin, nrows=0)
    nb_lignes = sum(len(morceau) for morceau in pd.read_csv(chemin, usecols=[0], chunksize=1_000_000))
    inventaire.append({'Fichier': chemin.name, 'Lignes': nb_lignes, 'Colonnes': len(en_tete.columns)})
df_inventaire = pd.DataFrame(inventaire)
df_inventaire


## 1.2 Charger et Compter les Données

Cette étape consiste à cartographier notre jeu de données en identifiant la structure de chaque table (`train.csv`, `oil.csv`, `stores.csv`, etc.). L'objectif est double : valider la bonne accessibilité des fichiers et mesurer leur volume avant d'entamer la phase d'analyse approfondie.

### Objectifs de la section
* **Vérification des fichiers :** S'assurer que tous les datasets indispensables (données d'entraînement, fonctionnalités économiques, calendriers et référentiels) sont présents dans le répertoire de travail.
* **Évaluation de la volumétrie :** Compter le nombre exact de lignes de chaque fichier pour anticiper les besoins en mémoire vive (RAM), en particulier pour la table principale `train.csv` qui stocke l'historique des ventes.
* **Extraction de la structure :** Lister les variables (colonnes) disponibles pour chaque table afin de préparer les futures jointures de données.

### Choix techniques et Optimisation de la mémoire
Le chargement de données massives de séries temporelles peut rapidement saturer la mémoire du Notebook et faire crasher le noyau (*Kernel*). Pour éviter cela, le script applique deux optimisations strictes :

1. **Lecture sélective des métadonnées (`nrows=0`) :** Nous chargeons uniquement la ligne d'en-tête pour obtenir instantanément le nom des colonnes sans lire la moindre ligne de données.
2. **Comptage chirurgical (`usecols` & `pyarrow`) :** Au lieu de charger des millions de lignes en mémoire pour les compter, le script extrait uniquement la toute première colonne. L'utilisation du moteur **PyArrow** (`engine='pyarrow'`) permet d'exécuter ce comptage de manière quasi instantanée, divisant le temps de calcul par dix pour la table `train.csv`.

In [ ]:
import os
import pandas as pd

# 1. Vérification du dossier de données
dossier_donnees = DOSSIER

# 2. Définition stricte des 7 tables
tables = [
    "train.csv",
    "horizon.csv",
    "items.csv",
    "stores.csv",
    "holidays_events.csv",
    "oil.csv",
    "transactions.csv"
]

# Dictionnaire pour stocker proprement toutes les tables
dfs = {}

print(" Début du chargement des tables indépendantes...\n")

for fichier in tables:
    fichier = fichier.strip()
    chemin_complet = os.path.join(dossier_donnees, fichier)
    
    # Extraire le nom de la variable (ex: "train.csv" devient "train")
    nom_table = fichier.replace('.csv', '')
    
    if os.path.isfile(chemin_complet):
        # ICI : On charge toute la table en mémoire pour pouvoir la manipuler plus tard
        dfs[nom_table] = pd.read_csv(chemin_complet)
        print(f" Table '{nom_table}' chargée avec succès ! Dimensions : {dfs[nom_table].shape}")
    else:
        print(f" Impossible de charger {fichier} (Introuvable)")

print("\n" + "="*50)
print(" Terminé ! Toutes les tables sont prêtes.")
print("="*50)

# 3. Rendre les variables accessibles directement par leur nom
train = dfs.get('train')
horizon = dfs.get('horizon')
items = dfs.get('items')
stores = dfs.get('stores')
holidays_events = dfs.get('holidays_events')
oil = dfs.get('oil')
transactions = dfs.get('transactions')

## La Jointure des tables pour former une nouvelle table regroupant toutes les tables et les manipuler facilement.

### Objectif
 Utiliser une fonction de pandas pour faire la jointure des tables avec une fonction de pandas qui est (`pd.merge`) . Cela amène à trouver la colonne en commun avec la table principal `train`et les joindres à partir de cela.

In [ ]:
df_complet = train.copy()
print(f"le tableau initial (train) :{df_complet.shape}")

# fusion avec les magasins (stores)
# clé de jointure : `store_nbr`
if stores is not None:
    df_complet = pd.merge(df_complet, stores, on = 'store_nbr', how = 'left')
    print(f"Après ajout de stores : {df_complet.shape}" )


# fusion avec les produits (items)
# clé de jointure : `item_nbr`
if items is not None:
    df_complet  = pd.merge(df_complet, items, on ='item_nbr', how = 'left')
    print(f"Après ajout de items : {df_complet.shape}")


# fusion avec le prix du pétrole (oil)
# clé de jointure : `date`
if oil is not None :
    df_complet['date'] = df_complet['date'].astype(str)
    oil_copy = oil.copy()
    oil_copy['date'] = oil_copy['date'].astype(str)

    df_complet = pd.merge(df_complet, oil_copy, on = 'date', how = 'left')
    print(f"Après ajout de oil : {df_complet.shape}")

# ==========================================
# FUSION AVEC LA TABLE HORIZON (CORRIGÉE)
# ==========================================
if horizon is not None:
    horizon_copy = horizon.drop_duplicates(subset=['store_nbr'], keep='first')
    
    # S'assurer que le 'date' de horizon ne va pas écraser votre 'date' principale
    if 'date' in horizon_copy.columns:
        horizon_copy = horizon_copy.rename(columns={'date': 'date_horizon'})

    df_complet = pd.merge(df_complet, horizon_copy, on='store_nbr', how='left')
    print(f"Après ajout de horizon : {df_complet.shape}")


# ==========================================
# FUSION AVEC LES TRANSACTIONS (CORRIGÉE)
# ==========================================
if transactions is not None:
    transactions_copy = transactions.copy()
    
    # Convertir en string comme pour la table oil pour éviter un conflit de type (str vs datetime)
    transactions_copy['date'] = transactions_copy['date'].astype(str)

    # Suppression des doublons sur le COUPLE (date, magasin)
    transactions_copy = transactions_copy.drop_duplicates(subset=['date', 'store_nbr'], keep='first')

    # Fusion
    df_complet = pd.merge(df_complet, transactions_copy, on=['date', 'store_nbr'], how='left')
    print(f"Après ajout de transactions : {df_complet.shape}")




# fusion avec les jours fériés (holidays_events)
# clé de la jointure :`date`
if holidays_events is not None :
   holidays_copy  = holidays_events.copy()
   holidays_copy['date'] = holidays_copy['date'].astype(str)

   holidays_copy = holidays_copy.drop_duplicates(subset = ['date'], keep = 'first')

   df_complet = pd.merge(df_complet, holidays_copy, on = 'date', how = 'left')
   print(f"Après ajout de holidays_events : {df_complet.shape}")

print(f"Aperçu de la table fusionnée :")
display(df_complet.head())

# 1.7 Doublons

Deux contrôles : les lignes strictement identiques, puis les doublons sur la clé `(date, store_nbr, item_nbr)`, qui doit être unique dans `train`.

In [ ]:
cle = ['date', 'store_nbr', 'item_nbr']
print('lignes strictement identiques :', int(train.duplicated().sum()))
print('doublons sur la clé (date, magasin, article) :', int(train.duplicated(subset=cle).sum()))


**Décision** : à compléter après exécution.

- Nombre de lignes retirées : ...
- Raison : ...

Si les doublons sont identiques sur toutes les colonnes, les retirer ne perd aucune information. Si la clé se répète avec des valeurs différentes, ce n'est pas une simple duplication : il faut comprendre d'où elle vient avant de choisir.

# Contrôle complémentaire : fuites

In [ ]:
# Remplacer 'sales' par le nom exact de votre variable cible
cible = 'unit_sales' 

if cible in df_complet.columns:
    print("---ANALYSE DES FUITES POTENTIELLES---")
    
    # Étape A : Trouver les corrélations suspectes (> 0.90)
    # Les colonnes numériques trop corrélées à la cible cachent souvent une fuite
    numerics = df_complet.select_dtypes(include=['int16', 'Int16', 'int32', 'Int32', 'float32', 'float64'])
    correlations = numerics.corr()[cible].abs().sort_values(ascending=False)
    
    print("\n[1] Top des colonnes suspectes par corrélation :")
    print(correlations[correlations > 0.85])
    
    # Étape B : Vérification des variables futures
    # Si 'transactions' ou 'dcoilwtico' changent de comportement selon la chronologie
    print("\n[2] Vérification des dates minimales et maximales par table :")
    print(f"Période globale du jeu de données : {df_complet['date'].min()} à {df_complet['date'].max()}")
else:
    print(f"La colonne cible '{cible}' n'est pas présente dans df_complet.")

# 1.8 Informations indisponibles au moment de la décision

La prévision porte sur 16 jours après la dernière date connue. Une colonne n'est utilisable que si sa valeur est connue à ce moment-là, pour les dates de `horizon`.

| Table | Colonne | Décision | Raison |
|---|---|---|---|
| train | `unit_sales` | écartée | c'est la cible |
| train | `id` | écartée | identifiant de ligne, sans information |
| train | `onpromotion` | gardée sous condition | planifiée à l'avance, à vérifier qu'elle est remplie dans `horizon` |
| horizon | `id` | écartée | identifiant de ligne |
| transactions | `transactions` | écartée | nombre de tickets du jour, inconnu pour l'horizon ; utilisable seulement en valeur retardée |
| oil | `dcoilwtico` | écartée | prix du jour, inconnu pour l'horizon ; utilisable seulement en valeur retardée |
| holidays_events | `type`, `locale`, `description` | gardées | calendrier connu d'avance |
| holidays_events | `transferred` | gardée avec réserve | peut avoir été corrigée après coup |
| stores | `city`, `state`, `type`, `cluster` | gardées | caractéristiques fixes |
| items | `family`, `class`, `perishable` | gardées | caractéristiques fixes |
| train / horizon | `date`, `store_nbr`, `item_nbr` | gardées | clés, connues pour l'horizon |

Contrôle à faire : `onpromotion` doit être remplie dans `horizon`, sinon elle ne peut pas servir.

# 2. Métrique d'évaluation

**Choix : NWRMSLE** (*normalized weighted root mean squared logarithmic error*), la métrique de la compétition Favorita.

- **Logarithme** : l'erreur porte sur le rapport entre prévision et réel, et non sur l'écart absolu. Un écart de 10 unités compte beaucoup sur un article qui se vend 2 fois par mois, et peu sur un article qui se vend 500 fois par jour. Les articles et magasins sont ainsi comparables entre eux.
- **Pondération** : les articles périssables comptent 1,25 fois plus que les autres, car une rupture ou un surstock y coûte plus cher (produits frais qui périment).
- **Retours** : les ventes négatives sont ramenées à 0 avant le calcul, car le logarithme n'est pas défini pour des valeurs négatives.
- **Normalisation** : on divise par la somme des poids, ce qui permet de comparer des jeux d'évaluation de tailles différentes.

Les métriques RMSE et MAE donneraient trop de poids aux gros volumes ; le MAPE est inutilisable avec des ventes nulles (division par zéro).

**Ce qu'elle ignore** : le coût économique réel de chaque erreur (un surstock de produit périssable n'est pas traité différemment d'un surstock de conserve), et les grosses ventes, dont l'erreur est écrasée par le logarithme.

**Pourquoi elle plutôt qu'une autre** : le RMSE serait dominé par les gros volumes, le MAE ignore la différence entre 1 et 100 ventes, et le MAPE explose sur les zéros.


In [ ]:
import numpy as np

def nwrmsle(y_vrai, y_pred, perissable=None):
    """NWRMSLE : racine de l'erreur quadratique logarithmique moyenne, pondérée.

    y_vrai, y_pred : ventes réelles et prévues (même longueur)
    perissable     : 1 si l'article est périssable, 0 sinon (None = poids égaux)
    """
    y_vrai = np.clip(np.asarray(y_vrai, dtype=float), 0, None)   # retours -> 0
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    if perissable is None:
        poids = np.ones_like(y_vrai)
    else:
        poids = np.where(np.asarray(perissable) == 1, 1.25, 1.0)
    erreur = (np.log1p(y_pred) - np.log1p(y_vrai)) ** 2
    return float(np.sqrt(np.sum(poids * erreur) / np.sum(poids)))


# Contrôles rapides : une prévision parfaite donne 0, et prévoir toujours 0 est une référence pénalisée
_reel = np.array([0, 3, 10, -1, 25])
print("prévision parfaite :", nwrmsle(_reel, _reel))
print("prévision nulle    :", round(nwrmsle(_reel, np.zeros(5)), 4))
print("avec périssables   :", round(nwrmsle(_reel, np.full(5, 5.0), perissable=[1, 0, 1, 0, 1]), 4))